# LeetCode 2279 — Maximum Bags With Full Capacity of Rocks bằng Python

Link đề: https://leetcode.com/problems/maximum-bags-with-full-capacity-of-rocks/ · Độ khó: **Medium** · Chủ đề: Array, Greedy, Sorting

Skeleton LeetCode:

```python
class Solution:
    def maximumBags(self, capacity: List[int], rocks: List[int], additionalRocks: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Rút gọn dữ liệu: mỗi túi chỉ còn một con số quan trọng — **lượng đá còn thiếu** `need = capacity − rocks`.
2. Nhận dạng bài **tham lam "làm việc rẻ trước"**: muốn làm đầy **nhiều túi nhất** với ngân sách cố định ⇒ lấp các túi thiếu ít nhất trước.
3. Chứng minh tham lam bằng **lập luận đổi chỗ** và bằng "tập k phần tử nhỏ nhất có tổng nhỏ nhất".
4. Cài đặt: vét cạn tập con, sắp xếp + quét, heap, tìm nhị phân trên số túi với tổng tiền tố, và **chọn nhanh** (quickselect) để đạt O(n) kỳ vọng.
5. Biết khi nào tham lam **không** còn đúng (mỗi túi có giá trị khác nhau ⇒ cái túi / knapsack).

## 2. Đề bài & ví dụ

Có `n` túi. Túi `i` chứa tối đa `capacity[i]` viên đá và hiện có `rocks[i]` viên. Bạn có thêm `additionalRocks` viên để bỏ vào **bất kỳ** túi nào.

Trả về **số túi đầy nhất** (túi có số đá = sức chứa) có thể đạt được.

**Ví dụ 1:** `capacity = [2,3,4,5]`, `rocks = [1,2,4,4]`, `additionalRocks = 2` → `3`. Lượng thiếu: `[1, 1, 0, 1]`. Túi 2 đã đầy; dùng 2 viên lấp túi 0 và 1 ⇒ 3 túi đầy.

**Ví dụ 2:** `capacity = [10,2,2]`, `rocks = [2,2,0]`, `additionalRocks = 100` → `3` (dư đá không sao).

**Ràng buộc:** `1 ≤ n ≤ 5·10⁴`, `1 ≤ capacity[i] ≤ 10⁹`, `0 ≤ rocks[i] ≤ capacity[i]`, `1 ≤ additionalRocks ≤ 10⁹`.

In [1]:
from typing import List
from itertools import accumulate, combinations
from bisect import bisect_right
import heapq, random, time

examples = [([2, 3, 4, 5], [1, 2, 4, 4], 2, 3), ([10, 2, 2], [2, 2, 0], 100, 3)]
for cap, rk, add, exp in examples:
    print("thiếu:", [c - r for c, r in zip(cap, rk)], "| thêm:", add, "| kỳ vọng", exp)

thiếu: [1, 1, 0, 1] | thêm: 2 | kỳ vọng 3
thiếu: [8, 0, 2] | thêm: 100 | kỳ vọng 3


## 3. Kiến thức nền

### 3.1 Ghép hai mảng bằng `zip` để tính lượng thiếu

In [2]:
cap, rk = [2, 3, 4, 5], [1, 2, 4, 4]
need = [c - r for c, r in zip(cap, rk)]
print(need, sorted(need))

[1, 1, 0, 1] [0, 1, 1, 1]


### 3.2 Tổng tiền tố của dãy đã sắp xếp

`pre[k]` = tổng `k` lượng thiếu nhỏ nhất = số đá tối thiểu để làm đầy `k` túi (sẽ chứng minh ở mục 6).

In [3]:
pre = list(accumulate(sorted([5, 1, 3, 0, 2]), initial=0))
print(pre)

[0, 0, 1, 3, 6, 11]


### 3.3 `bisect_right` trên dãy tiền tố không giảm

Số túi tối đa = số `k` lớn nhất với `pre[k] ≤ additionalRocks` = `bisect_right(pre, add) − 1`.

In [4]:
print([bisect_right(pre, a) - 1 for a in [0, 1, 3, 6, 100]])

[1, 2, 3, 4, 5]


### 3.4 `heapq.nsmallest` và heap

Lấy `k` phần tử nhỏ nhất mà không sắp xếp toàn bộ; hoặc `heapify` rồi `heappop` dần.

In [5]:
h = [5, 1, 3, 0, 2]; heapq.heapify(h)
print([heapq.heappop(h) for _ in range(3)], heapq.nsmallest(3, [5, 1, 3, 0, 2]))

[0, 1, 2] [0, 1, 2]


### 3.5 `sorted(generator)` — tạo và sắp xếp trong một bước

`sorted(c - r for c, r in zip(capacity, rocks))` tạo danh sách mới đã sắp xếp mà không cần list trung gian, và **không** sửa `capacity`/`rocks`. Timsort tận dụng các đoạn đã có thứ tự sẵn, nên nếu lượng thiếu gần như đã sắp xếp thì chạy gần O(n).

In [6]:
import time as _t
data = list(range(10**5))
t0 = _t.perf_counter(); sorted(data); t1 = _t.perf_counter() - t0
rnd = data[:]; random.Random(0).shuffle(rnd)
t0 = _t.perf_counter(); sorted(rnd); t2 = _t.perf_counter() - t0
print(f"sắp xếp dữ liệu đã có thứ tự: {t1*1000:.1f} ms | dữ liệu xáo trộn: {t2*1000:.1f} ms")

sắp xếp dữ liệu đã có thứ tự: 1.1 ms | dữ liệu xáo trộn: 15.5 ms


## 4. Ý tưởng nền tảng / trực giác

Mỗi túi `i` cần thêm đúng `need[i]` viên để đầy (bỏ thêm nữa là vô ích, bỏ ít hơn thì không đầy). Vậy câu hỏi trở thành:

> Chọn **nhiều phần tử nhất** từ mảng `need` sao cho **tổng** của chúng ≤ `additionalRocks`.

Trực giác: muốn chọn **nhiều** món với **ngân sách cố định** ⇒ chọn các món **rẻ nhất**. Sắp xếp `need` tăng dần rồi lấy lần lượt cho tới khi hết ngân sách.

Tính tay ví dụ 1: `need = [1, 1, 0, 1]` ⇒ sắp xếp `[0, 1, 1, 1]`, ngân sách 2:

| bước | need | ngân sách trước | lấp được? | ngân sách sau | số túi |
|---|---|---|---|---|---|
| 1 | 0 | 2 | ✔ | 2 | 1 |
| 2 | 1 | 2 | ✔ | 1 | 2 |
| 3 | 1 | 1 | ✔ | 0 | 3 |
| 4 | 1 | 0 | ✘ | — | 3 |

Túi đã đầy sẵn (`need = 0`) luôn được tính, không tốn đá.

In [7]:
need = sorted([1, 1, 0, 1]); left = 2; cnt = 0
for x in need:
    ok = x <= left
    print(f"need={x}, còn {left} -> {'lấp' if ok else 'không đủ'}")
    if not ok: break
    left -= x; cnt += 1
print("số túi đầy:", cnt)

need=0, còn 2 -> lấp
need=1, còn 2 -> lấp
need=1, còn 1 -> lấp
need=1, còn 0 -> không đủ
số túi đầy: 3


### 4.1 Vì sao không nên lấp "túi to" hay "túi gần đầy theo tỉ lệ"?

Một số người nghĩ theo **tỉ lệ** (túi 90% đầy trước) hoặc theo **sức chứa** (túi nhỏ trước). Nhưng mục tiêu chỉ đếm **số túi**, mỗi túi đóng góp **1** bất kể to nhỏ; chi phí của túi là **số đá còn thiếu** tuyệt đối. Túi sức chứa 10⁹ thiếu 1 viên "rẻ" hơn túi sức chứa 3 thiếu 2 viên.

In [8]:
cap2, rk2 = [10**9, 3], [10**9 - 1, 1]
print("thiếu:", [c - r for c, r in zip(cap2, rk2)], "| tỉ lệ đầy:", [r / c for c, r in zip(cap2, rk2)])

thiếu: [1, 2] | tỉ lệ đầy: [0.999999999, 0.3333333333333333]


## 5. Cách 1 — Brute Force / baseline: thử mọi tập túi

In [9]:
def brute(capacity, rocks, additionalRocks):
    need = [c - r for c, r in zip(capacity, rocks)]; n = len(need)
    for k in range(n, -1, -1):                      # thử số túi từ nhiều tới ít
        for comb in combinations(range(n), k):
            if sum(need[i] for i in comb) <= additionalRocks:
                return k
    return 0

for cap, rk, add, exp in examples:
    got = brute(cap, rk, add)
    print(got, "OK" if got == exp else "SAI")

3 OK
3 OK


**Độ phức tạp:** worst O(2ⁿ · n) — n = 20 đã ~2·10⁷.

**Nút thắt:** ta thử mọi tập kích thước `k`, trong khi chỉ cần kiểm tra **một** tập: `k` phần tử nhỏ nhất — nếu tập đó vượt ngân sách thì mọi tập kích thước `k` khác cũng vượt.

## 6. Ý tưởng thuật toán / bước đột phá

**Bổ đề:** trong mọi tập `k` phần tử của `need`, tập `k` phần tử **nhỏ nhất** có tổng **nhỏ nhất** (`pre[k]` sau khi sắp xếp).

⇒ Làm đầy được `k` túi ⇔ `pre[k] ≤ additionalRocks`. Vì `pre` không giảm, tập các `k` khả thi là `{0, 1, …, K}` ⇒ đáp án `K` = `max{k : pre[k] ≤ add}`.

**Lập luận đổi chỗ (cho tham lam):** giả sử một lời giải tối ưu lấp túi `x` nhưng bỏ qua túi `y` với `need[y] < need[x]`. Đổi: lấp `y` thay cho `x` ⇒ số túi giữ nguyên, số đá dùng giảm ⇒ vẫn hợp lệ. Lặp lại ⇒ có lời giải tối ưu gồm các túi thiếu ít nhất.

**Quy tắc:**
- Sắp `need` tăng dần; **nếu** `need[i] ≤ còn_lại` **thì** lấp, trừ ngân sách; **ngược lại** dừng (mọi túi sau thiếu nhiều hơn).

**Tăng tốc lý thuyết:** chỉ cần biết `K` — không cần sắp xếp hoàn toàn. Dùng **quickselect** tìm ngưỡng, hoặc tìm nhị phân trên `k` với "tổng `k` phần tử nhỏ nhất" (cần chọn nhanh) ⇒ O(n) kỳ vọng.

In [10]:
need = [7, 2, 9, 4, 1]
for k in range(len(need) + 1):
    best = min((sum(c) for c in combinations(need, k)), default=0)
    print(f"k={k}: tổng nhỏ nhất qua mọi tập = {best}, tổng k phần tử nhỏ nhất = {sum(sorted(need)[:k])}")

k=0: tổng nhỏ nhất qua mọi tập = 0, tổng k phần tử nhỏ nhất = 0
k=1: tổng nhỏ nhất qua mọi tập = 1, tổng k phần tử nhỏ nhất = 1
k=2: tổng nhỏ nhất qua mọi tập = 3, tổng k phần tử nhỏ nhất = 3
k=3: tổng nhỏ nhất qua mọi tập = 7, tổng k phần tử nhỏ nhất = 7
k=4: tổng nhỏ nhất qua mọi tập = 14, tổng k phần tử nhỏ nhất = 14
k=5: tổng nhỏ nhất qua mọi tập = 23, tổng k phần tử nhỏ nhất = 23


### 6.1 Đáp án đơn điệu ⇒ cũng có thể **tìm nhị phân trên số túi**

Hàm `f(k) = [tổng k lượng thiếu nhỏ nhất ≤ add]` đúng với `k ≤ K` và sai với `k > K`. Nếu có thao tác "tổng `k` phần tử nhỏ nhất" (ví dụ `heapq.nsmallest`), tìm nhị phân `k` cho đáp án — một kỹ thuật tổng quát (binary search on answer) dù ở đây sắp xếp đã đủ.

In [11]:
def binary_search_k(capacity, rocks, add):
    need = [c - r for c, r in zip(capacity, rocks)]
    lo, hi = 0, len(need)
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if sum(heapq.nsmallest(mid, need)) <= add: lo = mid
        else: hi = mid - 1
    return lo

for cap, rk, add, exp in examples:
    print(binary_search_k(cap, rk, add), "kỳ vọng", exp)

3 kỳ vọng 3
3 kỳ vọng 3


Độ phức tạp `O(n log n · log n)` — kém hơn sắp xếp một lần, nhưng minh họa rằng "tối đa số lượng với điều kiện đơn điệu" luôn có thể tìm nhị phân.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sắp xếp + quét (cách doocs)

In [12]:
def sort_greedy(capacity, rocks, additionalRocks):
    need = sorted(c - r for c, r in zip(capacity, rocks))
    for i, x in enumerate(need):
        if x > additionalRocks:
            return i
        additionalRocks -= x
    return len(need)

for cap, rk, add, exp in examples:
    got = sort_greedy(cap, rk, add)
    print(got, "OK" if got == exp else "SAI")

3 OK
3 OK


**Độ phức tạp cách 2:** O(n log n) thời gian (sắp xếp chiếm ưu thế), O(n) bộ nhớ cho mảng `need`.

### 7.2 Cách 3 — Tiền tố + tìm nhị phân

In [13]:
def prefix_bisect(capacity, rocks, additionalRocks):
    pre = list(accumulate(sorted(c - r for c, r in zip(capacity, rocks)), initial=0))
    return bisect_right(pre, additionalRocks) - 1

for cap, rk, add, exp in examples:
    got = prefix_bisect(cap, rk, add)
    print(got, "OK" if got == exp else "SAI")

3 OK
3 OK


Hữu ích khi có **nhiều truy vấn** ngân sách khác nhau trên cùng bộ túi: mỗi truy vấn O(log n).

### 7.3 Cách 4 — Heap (lấy dần túi thiếu ít nhất)

In [14]:
def heap_greedy(capacity, rocks, additionalRocks):
    h = [c - r for c, r in zip(capacity, rocks)]; heapq.heapify(h); cnt = 0
    while h and h[0] <= additionalRocks:
        additionalRocks -= heapq.heappop(h); cnt += 1
    return cnt

for cap, rk, add, exp in examples:
    print(heap_greedy(cap, rk, add), "OK" if heap_greedy(cap, rk, add) == exp else "SAI")

3 OK
3 OK


**Độ phức tạp cách 4:** O(n + K log n) — `heapify` O(n), mỗi lần lấy ra O(log n); tốt khi đáp án `K` nhỏ so với `n` (dừng sớm, không cần sắp xếp hết).

### 7.4 Cách 5 — Quickselect theo ngưỡng (O(n) kỳ vọng)

Ý tưởng: tìm giá trị ngưỡng `t` sao cho lấy mọi phần tử `< t` vẫn trong ngân sách, rồi lấy thêm bao nhiêu bản bằng `t` tùy ngân sách còn lại. Cài bằng chia ba nhóm `< pivot`, `= pivot`, `> pivot` với pivot ngẫu nhiên.

In [15]:
def quickselect_count(capacity, rocks, additionalRocks, seed=0):
    rnd = random.Random(seed)
    arr = [c - r for c, r in zip(capacity, rocks)]; budget = additionalRocks; cnt = 0
    while arr:
        p = arr[rnd.randrange(len(arr))]
        less = [x for x in arr if x < p]; eq = [x for x in arr if x == p]
        s_less = sum(less)
        if s_less > budget:
            arr = less                              # chưa lấy hết nhóm nhỏ: thu hẹp vào nhóm nhỏ
            continue
        budget -= s_less; cnt += len(less)          # lấy trọn nhóm nhỏ
        take = len(eq) if p == 0 else min(len(eq), budget // p)
        cnt += take; budget -= take * p
        if take < len(eq):
            return cnt                              # không lấy hết nhóm bằng: mọi phần tử lớn hơn cũng không lấy được
        arr = [x for x in arr if x > p]             # sang nhóm lớn hơn
    return cnt

for cap, rk, add, exp in examples:
    print(quickselect_count(cap, rk, add), "OK" if quickselect_count(cap, rk, add) == exp else "SAI")

3 OK
3 OK


Mỗi vòng loại bỏ (kỳ vọng) một phần hằng số của mảng ⇒ tổng O(n) kỳ vọng. Trong Python, `sorted` (C) thường vẫn nhanh hơn vì hằng số — cách này mang ý nghĩa thuật toán.

## 8. TRACE từng bước

Trace cách 2 trên một bộ dữ liệu lớn hơn (tự dựng): `capacity = [7, 3, 9, 4, 6, 5]`, `rocks = [2, 3, 1, 2, 6, 1]`, `additionalRocks = 10`.

In [16]:
def trace(capacity, rocks, add):
    need = sorted((c - r, i) for i, (c, r) in enumerate(zip(capacity, rocks)))
    print("lượng thiếu (đã sắp, kèm chỉ số túi):", need)
    cnt = 0
    for x, i in need:
        if x > add:
            print(f"túi {i} thiếu {x} > còn {add} -> dừng"); break
        add -= x; cnt += 1
        print(f"lấp túi {i} (thiếu {x}) -> còn {add} viên, đầy {cnt} túi")
    return cnt

print("Kết quả:", trace([7, 3, 9, 4, 6, 5], [2, 3, 1, 2, 6, 1], 10))

lượng thiếu (đã sắp, kèm chỉ số túi): [(0, 1), (0, 4), (2, 3), (4, 5), (5, 0), (8, 2)]
lấp túi 1 (thiếu 0) -> còn 10 viên, đầy 1 túi
lấp túi 4 (thiếu 0) -> còn 10 viên, đầy 2 túi
lấp túi 3 (thiếu 2) -> còn 8 viên, đầy 3 túi
lấp túi 5 (thiếu 4) -> còn 4 viên, đầy 4 túi
túi 0 thiếu 5 > còn 4 -> dừng
Kết quả: 4


Trace cách 3 (tiền tố + nhị phân) cho cùng dữ liệu với nhiều ngân sách:

In [17]:
need = sorted(c - r for c, r in zip([7, 3, 9, 4, 6, 5], [2, 3, 1, 2, 6, 1]))
pre = list(accumulate(need, initial=0))
print("need:", need, "| pre:", pre)
for add in [0, 1, 2, 5, 6, 10, 11, 100]:
    print(f"ngân sách {add:>3}: k = bisect_right(pre, {add}) - 1 = {bisect_right(pre, add) - 1}")

need: [0, 0, 2, 4, 5, 8] | pre: [0, 0, 0, 2, 6, 11, 19]
ngân sách   0: k = bisect_right(pre, 0) - 1 = 2
ngân sách   1: k = bisect_right(pre, 1) - 1 = 2
ngân sách   2: k = bisect_right(pre, 2) - 1 = 3
ngân sách   5: k = bisect_right(pre, 5) - 1 = 3
ngân sách   6: k = bisect_right(pre, 6) - 1 = 4
ngân sách  10: k = bisect_right(pre, 10) - 1 = 4
ngân sách  11: k = bisect_right(pre, 11) - 1 = 5
ngân sách 100: k = bisect_right(pre, 100) - 1 = 6


## 9. Code nộp LeetCode

In [18]:
from typing import List

class Solution:
    def maximumBags(self, capacity: List[int], rocks: List[int], additionalRocks: int) -> int:
        need = sorted(c - r for c, r in zip(capacity, rocks))   # túi thiếu ít nhất trước
        for i, x in enumerate(need):
            if x > additionalRocks:
                return i                 # không lấp nổi túi này -> các túi sau càng không
            additionalRocks -= x
        return len(need)

In [19]:
sol = Solution()
for cap, rk, add, exp in examples:
    got = sol.maximumBags(cap, rk, add)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

3 | kỳ vọng 3 | OK
3 | kỳ vọng 3 | OK


## 10. Vì sao thuật toán đúng

**Bổ đề (tập k nhỏ nhất).** Sắp `need` tăng dần `a₁ ≤ a₂ ≤ … ≤ aₙ`. Với mọi tập chỉ số `S`, `|S| = k`: `Σ_{i∈S} need[i] ≥ a₁ + … + a_k`. *Chứng minh:* sắp các phần tử của `S` tăng dần `b₁ ≤ … ≤ b_k`; phần tử nhỏ thứ `j` của `S` ≥ phần tử nhỏ thứ `j` của toàn mảng, tức `b_j ≥ a_j` (vì có ít nhất `j` phần tử của mảng ≤ `b_j`). Cộng lại. ∎

**Định lý.** Đáp án = `K = max{k : a₁ + … + a_k ≤ add}`.
- `K` túi (những túi thiếu ít nhất) lấp được ⇒ đáp án ≥ K.
- Nếu lấp được `k > K` túi thì theo bổ đề `add ≥ a₁ + … + a_k > add` — mâu thuẫn ⇒ đáp án ≤ K.

Thuật toán quét tham lam dừng đúng tại `K` (lần đầu `a_{K+1} > ngân sách còn lại`, tức `pre[K+1] > add`). ∎

**Túi đã đầy:** `need = 0` đứng đầu sau sắp xếp và luôn được đếm (0 ≤ ngân sách).

**Vì sao không bao giờ cần bỏ đá "dở dang" vào một túi?** Bỏ đá vào túi mà không làm nó đầy chẳng đóng góp gì cho mục tiêu, chỉ tiêu tốn ngân sách. Vì vậy một lời giải tối ưu luôn có thể chuẩn hóa thành: mỗi túi hoặc được lấp **đúng đủ**, hoặc **không** nhận viên nào — đó chính là mô hình "chọn tập con với tổng chi phí ≤ ngân sách" ở mục 4.

**Khi nào tham lam này sẽ sai?** Khi các món có **giá trị khác nhau** (biến thể 15.2) hoặc khi chi phí của một món phụ thuộc vào món khác đã chọn. Ở bài gốc mọi túi đều có giá trị 1 và chi phí độc lập ⇒ tham lam an toàn.

## 11. Phân tích Time / Space Complexity

`n ≤ 5·10⁴`.

| Cách | Time | Space |
|---|---|---|
| 1. Mọi tập con | O(2ⁿ · n) | O(n) |
| 2. Sắp xếp + quét | O(n log n) | O(n) |
| 3. Tiền tố + nhị phân | O(n log n) + O(log n)/truy vấn | O(n) |
| 4. Heap | O(n + K log n) | O(n) |
| 5. Quickselect | O(n) kỳ vọng | O(n) |

Tổng lượng thiếu có thể tới `5·10⁴ · 10⁹ = 5·10¹³` — vượt `int32`; Python không lo.

In [20]:
rng = random.Random(2279)
n = 5 * 10**4
cap = [rng.randint(1, 10**9) for _ in range(n)]; rk = [rng.randint(0, c) for c in cap]
for add in [10**6, 10**9]:
    res = []
    for name, f in [("sort_greedy", sort_greedy), ("prefix_bisect", prefix_bisect), ("heap_greedy", heap_greedy),
                    ("quickselect", quickselect_count), ("Solution", Solution().maximumBags)]:
        t0 = time.perf_counter(); res.append(f(cap, rk, add)); dt = time.perf_counter() - t0
        print(f"add={add:<10} {name:<13} -> {res[-1]:<5} {dt*1000:6.1f} ms")
    assert len(set(res)) == 1

add=1000000    sort_greedy   -> 32      13.3 ms


add=1000000    prefix_bisect -> 32      15.9 ms
add=1000000    heap_greedy   -> 32       3.8 ms
add=1000000    quickselect   -> 32       6.9 ms
add=1000000    Solution      -> 32      12.6 ms
add=1000000000 sort_greedy   -> 875     12.0 ms
add=1000000000 prefix_bisect -> 875     12.1 ms


add=1000000000 heap_greedy   -> 875      3.0 ms
add=1000000000 quickselect   -> 875      4.6 ms


add=1000000000 Solution      -> 875      8.7 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [21]:
ALL = {"brute": brute, "binary_search_k": binary_search_k, "sort_greedy": sort_greedy, "prefix_bisect": prefix_bisect, "heap_greedy": heap_greedy,
       "quickselect": quickselect_count, "Solution": Solution().maximumBags}
edge = examples + [
    ([1], [1], 1, 1),                        # đã đầy
    ([5], [0], 4, 0),                        # không đủ
    ([5], [0], 5, 1),                        # vừa đủ
    ([3, 3, 3], [3, 3, 3], 1, 3),            # tất cả đã đầy
    ([10**9, 10**9], [0, 10**9 - 1], 10**9, 1),
    ([2, 2, 2, 2], [0, 1, 0, 1], 3, 2),     # need [2,1,2,1]: 1 + 1, còn 1 < 2
    ([1, 2, 3, 4, 5], [0, 0, 0, 0, 0], 6, 3),
]
for cap, rk, add, exp in edge:
    for name, f in ALL.items():
        c2, r2 = list(cap), list(rk)
        assert f(c2, r2, add) == exp, (name, cap, rk, add, f(cap, rk, add), exp)
        assert c2 == cap and r2 == rk
print("Edge case OK:", len(edge))

Edge case OK: 9


### 12.2 Đối chiếu ngẫu nhiên với vét cạn (n ≤ 12, seed cố định)

In [22]:
rng = random.Random(42)
for _ in range(400):
    n = rng.randint(1, 12)
    cap = [rng.randint(1, 20) for _ in range(n)]; rk = [rng.randint(0, c) for c in cap]
    add = rng.randint(1, 60)
    ref = brute(cap, rk, add)
    for name, f in ALL.items():
        assert f(list(cap), list(rk), add) == ref, (name, cap, rk, add)
print("400 case ngẫu nhiên OK")

400 case ngẫu nhiên OK


### 12.3 Tính chất: đáp án không giảm khi tăng ngân sách; số túi đã đầy là cận dưới

In [23]:
rng = random.Random(7)
for _ in range(300):
    n = rng.randint(1, 50); cap = [rng.randint(1, 30) for _ in range(n)]; rk = [rng.randint(0, c) for c in cap]
    full = sum(1 for c, r in zip(cap, rk) if c == r); prev = -1
    for add in sorted(rng.sample(range(1, 500), 8)):
        v = Solution().maximumBags(cap, rk, add)
        assert v >= full and v >= prev; prev = v
print("tính chất OK")

tính chất OK


### 12.4 Cực trị theo ràng buộc

In [24]:
n = 5 * 10**4
cap = [10**9] * n; rk = [0] * n
assert Solution().maximumBags(cap, rk, 10**9) == 1           # chỉ lấp nổi một túi
assert Solution().maximumBags(cap, [10**9] * n, 1) == n       # mọi túi đã đầy
assert Solution().maximumBags([1] * n, [0] * n, 10**9) == n   # dư đá
print("cực trị OK")

cực trị OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Không sắp xếp (lấp theo thứ tự input)

In [25]:
def wrong_no_sort(capacity, rocks, add):
    cnt = 0
    for c, r in zip(capacity, rocks):
        if c - r <= add: add -= c - r; cnt += 1
        else: break                                      # SAI: dừng ở túi đắt đầu tiên theo thứ tự input
    return cnt

print("bản sai :", wrong_no_sort([10, 2, 2], [0, 1, 1], 2), "| bản đúng:", Solution().maximumBags([10, 2, 2], [0, 1, 1], 2))

bản sai : 0 | bản đúng: 2


**Sửa:** sắp xếp lượng thiếu tăng dần.

### Lỗi 2 — Sắp xếp theo `capacity` (hoặc theo `rocks`) thay vì lượng thiếu

In [26]:
def wrong_sort_capacity(capacity, rocks, add):
    cnt = 0
    for c, r in sorted(zip(capacity, rocks)):           # SAI: theo sức chứa
        if c - r > add: break
        add -= c - r; cnt += 1
    return cnt

print("bản sai :", wrong_sort_capacity([3, 100], [0, 99], 1), "| bản đúng:", Solution().maximumBags([3, 100], [0, 99], 1))

bản sai : 0 | bản đúng: 1


**Vì sao sai:** túi nhỏ (3) thiếu 3, túi lớn (100) chỉ thiếu 1. Chi phí là **lượng thiếu**, không phải sức chứa.

### Lỗi 3 — Không sắp xếp nhưng "bỏ qua" túi không đủ đá (vẫn sai)

In [27]:
def skip_instead_of_stop(capacity, rocks, add):
    cnt = 0
    for c, r in zip(capacity, rocks):                     # không sắp xếp, nhưng bỏ qua túi đắt
        if c - r <= add: add -= c - r; cnt += 1
    return cnt

print("bản 'bỏ qua' :", skip_instead_of_stop([3, 1, 1], [0, 0, 0], 3), "| bản đúng:", Solution().maximumBags([3, 1, 1], [0, 0, 0], 3))

bản 'bỏ qua' : 1 | bản đúng: 2


**Vì sao sai:** "bỏ qua túi không đủ, lấp túi nào đủ" vẫn là tham lam theo **thứ tự input**: túi 0 (thiếu 3) được lấp trước và tiêu hết ngân sách, mất hai túi thiếu 1. **Sửa:** sắp xếp; khi đã sắp xếp thì `break` và `continue` cho cùng kết quả (mọi túi sau đều đắt hơn).

### Lỗi 4 — So sánh `>=` / lệch dấu khi kiểm tra ngân sách

In [28]:
def wrong_strict(capacity, rocks, add):
    need = sorted(c - r for c, r in zip(capacity, rocks)); cnt = 0
    for x in need:
        if x >= add: break                                # SAI: vừa đủ vẫn lấp được
        add -= x; cnt += 1
    return cnt

print("bản sai :", wrong_strict([5], [0], 5), "| bản đúng:", Solution().maximumBags([5], [0], 5))

bản sai : 0 | bản đúng: 1


**Sửa:** lấp được khi `need ≤ còn lại` ⇒ dừng khi `need > còn lại`.

### Lỗi 5 — Sửa trực tiếp `rocks` / `capacity` của người gọi

In [29]:
def mutating(capacity, rocks, add):
    for i in range(len(rocks)):
        rocks[i] = capacity[i] - rocks[i]                 # tái dùng mảng rocks làm "need"
    rocks.sort(); cnt = 0
    for x in rocks:
        if x > add: break
        add -= x; cnt += 1
    return cnt

rk = [1, 2, 4, 4]
print("kết quả:", mutating([2, 3, 4, 5], rk, 2), "| rocks sau khi gọi:", rk, "(đã bị phá)")

kết quả: 3 | rocks sau khi gọi: [0, 1, 1, 1] (đã bị phá)


Trên LeetCode chấp nhận được (tiết kiệm bộ nhớ), nhưng trong code thật hãy tạo mảng mới.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tối đa **số lượng** món với ngân sách cố định, mỗi món có chi phí riêng, giá trị như nhau" ⇒ **sắp xếp chi phí tăng dần, lấy từ rẻ nhất**.
- Rút gọn dữ liệu về **một con số chi phí** cho mỗi phần tử trước khi nghĩ thuật toán (ở đây: `capacity − rocks`).
- Chứng minh nhanh bằng "**tập k nhỏ nhất có tổng nhỏ nhất**" hoặc đổi chỗ một cặp.
- Nhiều truy vấn ngân sách ⇒ tiền tố + `bisect`.
- Nếu các món có **giá trị khác nhau** ⇒ không còn tham lam: bài cái túi 0/1 (DP).
- Bài cùng họ: 1833 Maximum Ice Cream Bars, 1005, 2389 Longest Subsequence With Limited Sum (tiền tố + bisect), 455 Assign Cookies, 1710 Maximum Units on a Truck.

## 15. Biến thể của bài

### 15.1 Biến thể A — Nhiều truy vấn ngân sách (LC 2389 dạng tương tự)

In [30]:
def answer_queries(capacity, rocks, budgets):
    pre = list(accumulate(sorted(c - r for c, r in zip(capacity, rocks)), initial=0))
    return [bisect_right(pre, b) - 1 for b in budgets]

print(answer_queries([2, 3, 4, 5], [1, 2, 4, 4], [0, 1, 2, 3, 100]))

[1, 2, 3, 4, 4]


### 15.2 Biến thể B — Mỗi túi đầy cho **điểm** `w[i]` khác nhau: tối đa tổng điểm (knapsack 0/1)

Tham lam theo `need` không còn đúng; DP trên ngân sách (ngân sách nhỏ) O(n · B).

In [31]:
def max_value_full_bags(capacity, rocks, w, budget):
    dp = [0] * (budget + 1)
    for (c, r), val in zip(zip(capacity, rocks), w):
        cost = c - r
        for b in range(budget, cost - 1, -1):
            dp[b] = max(dp[b], dp[b - cost] + val)
    return dp[budget]

def greedy_by_need_value(capacity, rocks, w, budget):
    tot = 0
    for cost, val in sorted(((c - r), v) for (c, r), v in zip(zip(capacity, rocks), w)):
        if cost > budget: break
        budget -= cost; tot += val
    return tot

cap, rk, w = [3, 2, 2], [0, 0, 0], [10, 1, 1]
print("DP:", max_value_full_bags(cap, rk, w, 3), "| tham lam theo need:", greedy_by_need_value(cap, rk, w, 3), "(sai cho biến thể này)")

DP: 10 | tham lam theo need: 1 (sai cho biến thể này)


### 15.3 Biến thể C — Số đá **ít nhất** cần thêm để có ít nhất `k` túi đầy

Bằng tổng `k` lượng thiếu nhỏ nhất: `pre[k]` (hoặc −1 nếu `k > n`).

In [32]:
def min_rocks_for_k(capacity, rocks, k):
    need = sorted(c - r for c, r in zip(capacity, rocks))
    return -1 if k > len(need) else sum(need[:k])

print([min_rocks_for_k([2, 3, 4, 5], [1, 2, 4, 4], k) for k in range(6)])

[0, 0, 1, 2, 3, -1]


### 15.4 Biến thể D — Đá chỉ được thêm theo **gói** cố định `g` viên (mỗi túi nhận số gói nguyên)

Chi phí túi = `⌈need/g⌉` gói (thừa đá trong túi đầy? — giả sử túi chứa được thừa và vẫn tính là đầy khi ≥ sức chứa). Ngân sách tính theo gói ⇒ vẫn tham lam trên chi phí mới.

In [33]:
def max_bags_packs(capacity, rocks, packs, g):
    costs = sorted(-(-(c - r) // g) for c, r in zip(capacity, rocks)); cnt = 0
    for x in costs:
        if x > packs: break
        packs -= x; cnt += 1
    return cnt

print(max_bags_packs([2, 3, 4, 5], [1, 2, 4, 4], 2, 3), max_bags_packs([10, 7, 4], [0, 0, 0], 3, 4))

3 2


### 15.5 Biến thể E — Có thể **lấy bớt** đá từ túi này bỏ sang túi khác (không có đá thêm)

Tổng số đá cố định `R = Σ rocks`. Muốn nhiều túi đầy nhất ⇒ dồn đá vào các túi **sức chứa nhỏ nhất**: số túi tối đa = số `k` lớn nhất mà tổng `k` sức chứa nhỏ nhất ≤ `R` (giả sử đá chuyển tự do). Lại là tham lam "rẻ trước", với chi phí = sức chứa.

In [34]:
def max_full_redistribute(capacity, rocks):
    R = sum(rocks); cnt = 0
    for c in sorted(capacity):
        if c > R: break
        R -= c; cnt += 1
    return cnt

def redistribute_brute(capacity, rocks):
    R = sum(rocks); n = len(capacity)
    return max(k for k in range(n + 1) if any(sum(cb) <= R for cb in combinations(capacity, k)))

rng = random.Random(155)
for _ in range(300):
    cap = [rng.randint(1, 10) for _ in range(rng.randint(1, 8))]; rk = [rng.randint(0, c) for c in cap]
    assert max_full_redistribute(cap, rk) == redistribute_brute(cap, rk)
print(max_full_redistribute([2, 3, 4, 5], [1, 2, 4, 4]), "- khớp brute force")

3 - khớp brute force


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi tập con | đúng định nghĩa | O(2ⁿ·n) | O(n) | kiểm chứng |
| 2. Sắp xếp + quét | rẻ trước | O(n log n) | O(n) | **mặc định** |
| 3. Tiền tố + nhị phân | `bisect_right(pre, add)` | O(n log n) | O(n) | nhiều truy vấn |
| 4. Heap | lấy dần nhỏ nhất | O(n + K log n) | O(n) | K nhỏ |
| 5. Quickselect | chia ba theo pivot | O(n) kỳ vọng | O(n) | lý thuyết tuyến tính |
| 6.1 Nhị phân trên k | `nsmallest(k)` đơn điệu | O(n log² n) | O(n) | minh họa "binary search on answer" |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `maximumBags([5, 5, 5], [1, 2, 3], 5)` và `maximumBags([1, 1], [0, 0], 1)`?

**Bài 2 (tính tay):** Với `need = [4, 0, 2, 7, 1, 3]` và ngân sách 6, các túi nào được lấp?

**Bài 3 (điền TODO):** Viết `bt3_todo(capacity, rocks, additionalRocks)` trả về **danh sách chỉ số** các túi đầy trong một lời giải tối ưu (gồm cả túi đầy sẵn).

**Bài 4 (tự viết lại):** Viết `bt4_todo(...)` cài cách 4 (heap) không nhìn code.

**Bài 5 (mini challenge):** Túi được chia thành `m` **nhóm**; muốn tối đa **số nhóm có mọi túi đầy**. Viết `bt5_todo(capacity, rocks, groups, add)` với `groups[i]` là nhóm của túi `i`. Gợi ý: chi phí một nhóm = tổng lượng thiếu của nhóm ⇒ lại là bài gốc trên các nhóm.

In [35]:
def bt3_todo(capacity, rocks, additionalRocks):
    # TODO: chỉ số các túi đầy (tối ưu)
    pass

def bt4_todo(capacity, rocks, additionalRocks):
    # TODO: heap
    pass

def bt5_todo(capacity, rocks, groups, additionalRocks):
    # TODO: tối đa số nhóm đầy
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: thiếu `[4, 3, 2]`. Mức 2: sắp `[2, 3, 4]`; 2 + 3 = 5. Mức 3: `2`; ví dụ hai: thiếu `[1, 1]`, ngân sách 1 ⇒ `1`.

**Bài 2.** Mức 1: sắp `[0, 1, 2, 3, 4, 7]`. Mức 2: tiền tố `0, 1, 3, 6, 10`. Mức 3: lấp túi thiếu `0, 1, 2, 3` (tổng 6) ⇒ 4 túi (chỉ số gốc 1, 4, 2, 5).

**Bài 3.** Mức 1: sắp xếp kèm chỉ số. Mức 2: lấy dần. Mức 3: trả list chỉ số.

**Bài 4.** Mức 1: `heapify(need)`. Mức 2: `while h and h[0] <= add`. Mức 3: pop, trừ, đếm.

**Bài 5.** Mức 1: gom lượng thiếu theo nhóm (`defaultdict(int)`). Mức 2: mỗi nhóm thành một "túi lớn" với chi phí = tổng. Mức 3: sắp chi phí nhóm, tham lam.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `2`, `1`. **Bài 2:** các túi thiếu `0, 1, 2, 3` ⇒ 4 túi.

In [36]:
from collections import defaultdict
assert Solution().maximumBags([5, 5, 5], [1, 2, 3], 5) == 2
assert Solution().maximumBags([1, 1], [0, 0], 1) == 1

def bt3(capacity, rocks, additionalRocks):
    order = sorted(range(len(capacity)), key=lambda i: capacity[i] - rocks[i]); out = []
    for i in order:
        x = capacity[i] - rocks[i]
        if x > additionalRocks: break
        additionalRocks -= x; out.append(i)
    return out

def bt4(capacity, rocks, additionalRocks):
    return heap_greedy(capacity, rocks, additionalRocks)

def bt5(capacity, rocks, groups, additionalRocks):
    cost = defaultdict(int)
    for c, r, g in zip(capacity, rocks, groups): cost[g] += c - r
    cnt = 0
    for x in sorted(cost.values()):
        if x > additionalRocks: break
        additionalRocks -= x; cnt += 1
    return cnt

def bt5_brute(capacity, rocks, groups, add):
    cost = defaultdict(int)
    for c, r, g in zip(capacity, rocks, groups): cost[g] += c - r
    vals = list(cost.values()); best = 0
    for k in range(len(vals) + 1):
        if any(sum(cb) <= add for cb in combinations(vals, k)): best = k
    return best

need_demo = [4, 0, 2, 7, 1, 3]
assert bt3([x + 1 for x in need_demo], [1] * 6, 6) == [1, 4, 2, 5]
rng = random.Random(1979)
for _ in range(400):
    n = rng.randint(1, 10); cap = [rng.randint(1, 15) for _ in range(n)]; rk = [rng.randint(0, c) for c in cap]
    add = rng.randint(1, 40)
    lst = bt3(cap, rk, add)
    assert len(lst) == brute(cap, rk, add) and sum(cap[i] - rk[i] for i in lst) <= add
    assert bt4(cap, rk, add) == brute(cap, rk, add)
    grp = [rng.randint(0, 3) for _ in range(n)]
    assert bt5(cap, rk, grp, add) == bt5_brute(cap, rk, grp, add)
print("Đáp án bài tập: tất cả assert OK")

Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
need = sorted(c - r for c, r in zip(capacity, rocks))
for i, x in enumerate(need):
    if x > additionalRocks:
        return i
    additionalRocks -= x
return len(need)
```

**Checklist:**
- [ ] Chi phí = **lượng thiếu** `capacity − rocks` (không phải sức chứa / tỉ lệ).
- [ ] Sắp xếp **tăng dần**, lấy rẻ trước.
- [ ] Lấp được khi `need ≤ còn lại` (bằng vẫn được).
- [ ] Túi đã đầy (`need = 0`) luôn được đếm.
- [ ] Đã thử: ngân sách thừa, ngân sách vừa đủ, mọi túi đã đầy.
- [ ] Không sửa `capacity`/`rocks` của người gọi (dùng `sorted` tạo mảng mới).